# 蒙块选址可视化（结构化噪声）
**回答的问题：蒙块蒙的位置对不对、剂量多少。**

三联图：原图 | seg 输入梯度热力 | 蒙版叠加（蒙区染红）。
用模型自己的 mask_selector 现采，与训练时同一套选址逻辑。

In [ ]:
import os, sys
import torch

# ---- 路径：notebook 在 visual/ 下，切到 mmsegmentation 作为工作目录 ----
MMSEG = os.path.abspath('../tasks/Segmentation/mmsegmentation')
os.chdir(MMSEG)
sys_path_added = []
for p in [MMSEG, os.path.join(MMSEG, 'eden')]:
    import sys
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin, eden.pcgrad_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)
sys.modules.setdefault('mydiffae.pcgrad_mixin', eden.pcgrad_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

sys.path.insert(0, os.path.abspath('../visual'))
import probe_core as pc


In [ ]:
CFG = 'configs/fcn/fcn_r50-d8_4xb4-80k_ade20k-512x512_diffae_mask.py'
CKPT = 'work_dirs/eden/fcn_diffae_mask/iter_80000.pth'  # None = 不加载（看初始选址）
OUT = 'work_dirs/offline_probe/mask'
NUM_IMAGES = 4
os.makedirs(OUT, exist_ok=True)

In [ ]:
model, cfg = pc.build_model(CFG, CKPT)
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
sel = pc.measure_mask_selection(model, batch, NUM_IMAGES)
fig = pc.plot_mask_selection(sel, title='mask selection',
                             save_path=os.path.join(OUT, 'mask.png'))
fig